# Week 1: API Fundamentals, Chatbot & Structured Output

**Lab companion to [week_01.md](../agenda/week_01.md)**

In this lab, you will:
1. Make your first API call and understand tokens
2. Build a chatbot with conversation memory
3. Master prompt engineering and structured (JSON) output

## 1. Setup

First, install the required packages:

In [ ]:
# Run this once to install dependencies
!pip install openai python-dotenv

In [ ]:
# Import libraries
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load API key from .env file
load_dotenv()

# Create client
client = OpenAI()

print("✓ Setup complete!")

## 2. Your First API Call

Let's talk to GPT! The simplest possible call:

In [ ]:
# Make your first API call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Say hello!"}
    ]
)

# Print the response
print(response.choices[0].message.content)

## 3. Understanding the Response Object

The API returns a lot of useful information. Let's explore it:

In [ ]:
# Let's look at the full response structure
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is Python in one sentence?"}
    ]
)

print("Model used:", response.model)
print("Response ID:", response.id)
print("\n--- Token Usage ---")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print("\n--- The Answer ---")
print(response.choices[0].message.content)

## 4. Message Roles

There are three roles in a conversation:
- `system`: Sets the AI's behavior
- `user`: What you say
- `assistant`: What the AI says

In [ ]:
# Using a system message to change behavior
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a pirate. Respond in pirate speak."},
        {"role": "user", "content": "How do I learn Python?"}
    ]
)

print(response.choices[0].message.content)

## 5. Understanding Tokens

Tokens are how the AI "sees" text. Let's visualize this:

In [ ]:
# Install tiktoken for tokenization
!pip install tiktoken -q

In [ ]:
import tiktoken

# Get the tokenizer for GPT-4
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Example texts
texts = [
    "Hello",
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "Supercalifragilisticexpialidocious"
]

print("Text -> Token Count -> Tokens")
print("-" * 60)
for text in texts:
    tokens = encoding.encode(text)
    print(f"{text}")
    print(f"  → {len(tokens)} tokens: {tokens}")
    print()

## 6. Temperature: Controlling Creativity

Temperature controls randomness:
- `0.0` = Deterministic (same answer every time)
- `1.0` = More creative/varied

In [ ]:
# Low temperature (deterministic)
prompt = "Give me a creative name for a coffee shop."

print("Temperature 0 (deterministic):")
for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    print(f"  {i+1}. {response.choices[0].message.content}")

print("\nTemperature 1 (creative):")
for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1
    )
    print(f"  {i+1}. {response.choices[0].message.content}")

## 7. Controlling Output Length

Use `max_tokens` to limit response length:

In [ ]:
# Short response
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain machine learning."}],
    max_tokens=30
)
print("Max 30 tokens:")
print(response.choices[0].message.content)
print(f"(Used {response.usage.completion_tokens} tokens)")

# Longer response
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain machine learning."}],
    max_tokens=100
)
print("\nMax 100 tokens:")
print(response.choices[0].message.content)
print(f"(Used {response.usage.completion_tokens} tokens)")

## 8. Exercise: Build a Simple Helper Function

Create a reusable function to chat with the AI:

In [ ]:
def ask(question: str, system: str = "You are a helpful assistant.") -> str:
    """Simple helper to ask the AI a question."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content

# Test it!
print(ask("What is the capital of France?"))

In [ ]:
# Try different system prompts
print("As a teacher:")
print(ask("What is 2+2?", system="You are a math teacher. Explain your answer."))

print("\nAs a comedian:")
print(ask("What is 2+2?", system="You are a comedian. Make it funny."))

## 9. Calculating Costs

GPT-4o-mini pricing (as of 2024):
- Input: $0.15 per 1M tokens
- Output: $0.60 per 1M tokens

In [ ]:
def calculate_cost(response, model="gpt-4o-mini"):
    """Calculate the cost of an API call."""
    # Pricing per 1M tokens
    prices = {
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},
        "gpt-4o": {"input": 2.50, "output": 10.00},
    }

    p = prices.get(model, prices["gpt-4o-mini"])
    input_cost = (response.usage.prompt_tokens / 1_000_000) * p["input"]
    output_cost = (response.usage.completion_tokens / 1_000_000) * p["output"]

    return input_cost + output_cost

# Test
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a haiku about programming."}]
)

cost = calculate_cost(response)
print(f"Response: {response.choices[0].message.content}")
print(f"\nTokens: {response.usage.total_tokens}")
print(f"Cost: ${cost:.6f}")

## 10. Your Turn! Experiments

Try these exercises:

In [ ]:
# Exercise 1: Create a translator
# Fill in the system prompt to make the AI translate to Spanish

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "YOUR SYSTEM PROMPT HERE"},
        {"role": "user", "content": "Hello, how are you?"}
    ]
)
print(response.choices[0].message.content)

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

The system prompt tells the AI what role to play. For translation:
1. Tell it to act as a translator
2. Specify the target language (Spanish)
3. Tell it to only output the translation

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a translator. Translate to Spanish. Only output the translation."},
        {"role": "user", "content": "Hello, how are you?"}
    ]
)
print(response.choices[0].message.content)
# Output: Hola, ¿cómo estás?
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

The system prompt tells the AI what role to play. For translation:
1. Tell it to act as a translator
2. Specify the target language (Spanish)
3. Tell it to only output the translation

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a translator. Translate to Spanish. Only output the translation."},
        {"role": "user", "content": "Hello, how are you?"}
    ]
)
print(response.choices[0].message.content)
# Output: Hola, ¿cómo estás?
```

</details>

In [ ]:
# Exercise 2: Create a code explainer
# Make the AI explain code to a beginner

code = """
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Explain code simply to a beginner."},
        {"role": "user", "content": f"Explain this code:\n{code}"}
    ]
)
print(response.choices[0].message.content)

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

This one is already mostly complete! Try variations:
- Change to "Explain like I am 5"
- Add "Use analogies" to the prompt
- Ask for "step by step" explanation

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
# ELI5 version - great for beginners
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Explain code like I am 5. Use simple analogies."},
        {"role": "user", "content": f"What does this code do?\n{code}"}
    ]
)
print(response.choices[0].message.content)
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

This one is already mostly complete! Try variations:
- Change to "Explain like I am 5"
- Add "Use analogies" to the prompt
- Ask for "step by step" explanation

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
# ELI5 version - great for beginners
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Explain code like I am 5. Use simple analogies."},
        {"role": "user", "content": f"What does this code do?\n{code}"}
    ]
)
print(response.choices[0].message.content)
```

</details>

---

# Part 2: Chatbot with Memory

In [ ]:
# Setup
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

print("✓ Ready!")

## 1. The Problem: AI Has No Memory

Each API call is independent. The AI doesn't remember previous messages:

In [ ]:
# First message
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "My name is Alice."}]
)
print("AI:", response1.choices[0].message.content)

# Second message - new conversation, no memory!
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is my name?"}]
)
print("AI:", response2.choices[0].message.content)

## 2. The Solution: Send Conversation History

We must send all previous messages with each request:

In [ ]:
# Manual conversation history
messages = []

# First turn
messages.append({"role": "user", "content": "My name is Alice."})
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)
ai_reply = response.choices[0].message.content
messages.append({"role": "assistant", "content": ai_reply})
print("AI:", ai_reply)

# Second turn - now with history!
messages.append({"role": "user", "content": "What is my name?"})
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)
print("AI:", response.choices[0].message.content)

In [ ]:
# Let's see what we're sending
print("Conversation history:")
for msg in messages:
    print(f"  [{msg['role']}]: {msg['content'][:50]}...")

## 3. Build a Chatbot Class

Let's encapsulate this in a reusable class:

In [ ]:
class Chatbot:
    """A simple chatbot with conversation memory."""

    def __init__(self, system_prompt: str = "You are a helpful assistant."):
        self.client = OpenAI()
        self.system_prompt = system_prompt
        self.history = []

    def chat(self, message: str) -> str:
        """Send a message and get a response."""
        # Add user message to history
        self.history.append({"role": "user", "content": message})

        # Build messages with system prompt + history
        messages = [
            {"role": "system", "content": self.system_prompt}
        ] + self.history

        # Get response
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )

        # Add assistant response to history
        ai_reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": ai_reply})

        return ai_reply

    def reset(self):
        """Clear conversation history."""
        self.history = []

# Create and test
bot = Chatbot()
print(bot.chat("My name is Bob."))
print(bot.chat("What is my name?"))

In [ ]:
# Test with a custom personality
pirate_bot = Chatbot(system_prompt="You are a friendly pirate. Use pirate speak.")

print(pirate_bot.chat("Hello!"))
print()
print(pirate_bot.chat("What treasure are you seeking?"))

## 4. Problem: Context Window Limits

Models have token limits. If conversation gets too long, we need strategies:

In [ ]:
import tiktoken

def count_tokens(messages: list, model: str = "gpt-4o-mini") -> int:
    """Count tokens in a message list."""
    encoding = tiktoken.encoding_for_model(model)
    total = 0
    for msg in messages:
        total += 4  # Message overhead
        total += len(encoding.encode(msg.get("content", "")))
    return total

# Check our chatbot's token usage
print(f"Current conversation uses {count_tokens(bot.history)} tokens")
print(f"GPT-4o-mini context limit: 128,000 tokens")

## 5. Memory Strategy: Sliding Window

Keep only the last N messages:

In [ ]:
class SlidingWindowChatbot:
    """Chatbot that keeps only the last N messages."""

    def __init__(self, system_prompt: str = "You are helpful.", max_messages: int = 10):
        self.client = OpenAI()
        self.system_prompt = system_prompt
        self.history = []
        self.max_messages = max_messages

    def chat(self, message: str) -> str:
        self.history.append({"role": "user", "content": message})

        # Trim to max_messages
        if len(self.history) > self.max_messages:
            self.history = self.history[-self.max_messages:]

        messages = [{"role": "system", "content": self.system_prompt}] + self.history

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )

        ai_reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": ai_reply})

        return ai_reply

# Test - only keeps last 4 messages
bot = SlidingWindowChatbot(max_messages=4)
print(bot.chat("My name is Charlie."))
print(bot.chat("I like pizza."))
print(bot.chat("My favorite color is blue."))
print(bot.chat("I work as a doctor."))
# This might forget the name!
print("---")
print(bot.chat("What do you remember about me?"))

## 6. Memory Strategy: Summarization

Summarize old messages instead of discarding:

In [ ]:
class SummarizingChatbot:
    """Chatbot that summarizes old messages."""

    def __init__(self, max_messages: int = 6):
        self.client = OpenAI()
        self.history = []
        self.summary = ""
        self.max_messages = max_messages

    def _summarize(self, messages: list) -> str:
        """Summarize a list of messages."""
        conversation = "\n".join([
            f"{m['role']}: {m['content']}" for m in messages
        ])

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": f"Summarize this conversation briefly, keeping key facts:\n\n{conversation}"
            }]
        )
        return response.choices[0].message.content

    def chat(self, message: str) -> str:
        self.history.append({"role": "user", "content": message})

        # If too many messages, summarize the old ones
        if len(self.history) > self.max_messages:
            # Summarize first half
            to_summarize = self.history[:len(self.history)//2]
            new_summary = self._summarize(to_summarize)

            if self.summary:
                self.summary = f"{self.summary}\n\nMore recently: {new_summary}"
            else:
                self.summary = new_summary

            # Keep only recent messages
            self.history = self.history[len(self.history)//2:]

        # Build prompt with summary + recent history
        system = "You are a helpful assistant."
        if self.summary:
            system += f"\n\nPrevious conversation summary: {self.summary}"

        messages = [{"role": "system", "content": system}] + self.history

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )

        ai_reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": ai_reply})

        return ai_reply

# Test
bot = SummarizingChatbot(max_messages=4)
print(bot.chat("My name is Diana."))
print(bot.chat("I live in Paris."))
print(bot.chat("I work as a chef."))
print(bot.chat("My favorite dish is ratatouille."))
print("--- After summarization ---")
print("Summary:", bot.summary)
print()
print(bot.chat("What do you remember about me?"))

## 7. Interactive Chat Loop

Build a real interactive chatbot:

In [ ]:
def interactive_chat():
    """Run an interactive chat session."""
    bot = Chatbot(system_prompt="You are a friendly AI assistant named Aria.")

    print("Chat with Aria! (type 'quit' to exit)\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break

        response = bot.chat(user_input)
        print(f"Aria: {response}\n")

# Uncomment to run interactive chat
# interactive_chat()

## 8. Exercises

Try these challenges:

In [ ]:
# Exercise 1: Create a study buddy chatbot
# It should help explain concepts and quiz you

class StudyBuddy(Chatbot):
    def __init__(self):
        super().__init__(system_prompt="""
You are a study buddy. Help the user learn by:
1. Explaining concepts clearly
2. Asking follow-up questions to test understanding
3. Providing examples
4. Being encouraging
""")

buddy = StudyBuddy()
print(buddy.chat("Explain what a variable is in programming."))

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

A study buddy chatbot should:
1. Have a friendly, encouraging persona
2. Remember the topic being studied
3. Ask follow-up questions to test understanding

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class StudyBuddy(Chatbot):
    def __init__(self):
        super().__init__(
            system_prompt="""You are a friendly study buddy. Help the user learn by:
1. Explaining concepts clearly
2. Asking follow-up questions to test understanding
3. Giving encouragement
4. Suggesting related topics to explore"""
        )

buddy = StudyBuddy()
print(buddy.chat("Help me understand recursion"))
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

A study buddy chatbot should:
1. Have a friendly, encouraging persona
2. Remember the topic being studied
3. Ask follow-up questions to test understanding

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class StudyBuddy(Chatbot):
    def __init__(self):
        super().__init__(
            system_prompt="""You are a friendly study buddy. Help the user learn by:
1. Explaining concepts clearly
2. Asking follow-up questions to test understanding
3. Giving encouragement
4. Suggesting related topics to explore"""
        )

buddy = StudyBuddy()
print(buddy.chat("Help me understand recursion"))
```

</details>

In [ ]:
# Exercise 2: Add message count tracking
# How many messages have been exchanged?

class TrackedChatbot(Chatbot):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.message_count = 0
        self.total_tokens = 0

    def chat(self, message: str) -> str:
        self.message_count += 1
        return super().chat(message)

    def stats(self) -> dict:
        return {
            "messages": self.message_count,
            "history_length": len(self.history)
        }

bot = TrackedChatbot()
bot.chat("Hello!")
bot.chat("How are you?")
print("Stats:", bot.stats())

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Track the count in the chatbot class:
1. Add a counter attribute in __init__
2. Increment it in the chat() method
3. Add a method to get stats

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class ChatbotWithStats(Chatbot):
    def __init__(self, system_prompt="You are a helpful assistant."):
        super().__init__(system_prompt)
        self.message_count = 0
        self.total_tokens = 0

    def chat(self, message: str) -> str:
        self.message_count += 1
        response = super().chat(message)
        return response

    def get_stats(self):
        return {
            "messages": self.message_count,
            "history_length": len(self.history)
        }

bot = ChatbotWithStats()
bot.chat("Hello!")
bot.chat("How are you?")
print(bot.get_stats())  # {"messages": 2, ...}
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Track the count in the chatbot class:
1. Add a counter attribute in __init__
2. Increment it in the chat() method
3. Add a method to get stats

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class ChatbotWithStats(Chatbot):
    def __init__(self, system_prompt="You are a helpful assistant."):
        super().__init__(system_prompt)
        self.message_count = 0
        self.total_tokens = 0

    def chat(self, message: str) -> str:
        self.message_count += 1
        response = super().chat(message)
        return response

    def get_stats(self):
        return {
            "messages": self.message_count,
            "history_length": len(self.history)
        }

bot = ChatbotWithStats()
bot.chat("Hello!")
bot.chat("How are you?")
print(bot.get_stats())  # {"messages": 2, ...}
```

</details>

---

# Part 3: Better Prompts & Structured Output

In [ ]:
# Setup
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv()
client = OpenAI()

print("✓ Ready!")

## 1. Prompt Engineering Basics

### Zero-Shot: Just ask directly

In [ ]:
# Zero-shot classification
text = "This product is amazing! Best purchase ever!"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": f"Classify this text as positive, negative, or neutral: '{text}'"
    }]
)

print(response.choices[0].message.content)

### Few-Shot: Provide examples

In [ ]:
# Few-shot classification with examples
prompt = """Classify sentiment as positive, negative, or neutral.

Examples:
"I love this!" -> positive
"Terrible, don't buy" -> negative
"It's okay, nothing special" -> neutral

Now classify: "This was a waste of money"
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

### Chain-of-Thought: Make AI explain reasoning

In [ ]:
# Chain-of-thought for better reasoning
prompt = """Solve this step by step:

A store has 50 apples. They sell 23 in the morning and receive a delivery of 35 more.
In the afternoon, they sell 18 and throw away 5 that went bad.
How many apples do they have at the end of the day?

Think through each step before giving the final answer.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

## 2. Getting JSON Output

### Method 1: Ask for JSON (unreliable)

In [ ]:
# Just asking for JSON (sometimes fails)
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": "Extract the person's name and age from: 'John is 25 years old'. Return JSON."
    }]
)

print("Raw response:")
print(response.choices[0].message.content)

# Try to parse - might fail!
try:
    data = json.loads(response.choices[0].message.content)
    print("\nParsed:", data)
except json.JSONDecodeError as e:
    print(f"\nFailed to parse JSON: {e}")

### Method 2: JSON Mode (reliable)

In [ ]:
# Using response_format for guaranteed JSON
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": "Extract the person's name and age from: 'John is 25 years old'. Return as JSON with 'name' and 'age' fields."
    }],
    response_format={"type": "json_object"}
)

# Now it's always valid JSON
data = json.loads(response.choices[0].message.content)
print("Parsed:", data)
print(f"Name: {data['name']}, Age: {data['age']}")

## 3. Pydantic Validation

Make AI output match a specific schema:

In [ ]:
!pip install pydantic -q

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Define the schema
class Person(BaseModel):
    name: str = Field(description="The person's full name")
    age: int = Field(description="Age in years")
    occupation: Optional[str] = Field(default=None, description="Job title")

# Generate the schema as a string for the prompt
schema_str = Person.model_json_schema()
print("Schema:")
print(json.dumps(schema_str, indent=2))

In [ ]:
def extract_person(text: str) -> Person:
    """Extract person info from text using Pydantic validation."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Extract person information from this text:
{text}

Return JSON matching this schema:
- name: string (full name)
- age: integer
- occupation: string or null"""
        }],
        response_format={"type": "json_object"}
    )

    data = json.loads(response.choices[0].message.content)
    return Person(**data)  # Validates and converts

# Test
person = extract_person("Dr. Sarah Johnson, a 42-year-old surgeon, won the award.")
print(f"Name: {person.name}")
print(f"Age: {person.age}")
print(f"Occupation: {person.occupation}")

## 4. Complex Extraction

Extract multiple entities with nested structures:

In [ ]:
from typing import List
from enum import Enum

class Priority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

class Task(BaseModel):
    title: str
    priority: Priority
    assignee: Optional[str] = None

class MeetingNotes(BaseModel):
    title: str
    date: str
    attendees: List[str]
    tasks: List[Task]
    summary: str

def extract_meeting_notes(transcript: str) -> MeetingNotes:
    """Extract structured meeting notes from a transcript."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Extract meeting notes from this transcript:

{transcript}

Return JSON with:
- title: meeting title
- date: date mentioned (or "unknown")
- attendees: list of participant names
- tasks: list of {{title, priority (low/medium/high), assignee}}
- summary: brief 1-2 sentence summary"""
        }],
        response_format={"type": "json_object"}
    )

    data = json.loads(response.choices[0].message.content)
    return MeetingNotes(**data)

# Test
transcript = """
Project sync meeting, March 15th.
Attendees: Alice, Bob, and Carol.

Alice mentioned the frontend is almost done, just needs testing.
Bob said the API has a critical bug that needs fixing ASAP - he'll handle it.
Carol will review the documentation by end of week, low priority.

Overall, we're on track for the launch.
"""

notes = extract_meeting_notes(transcript)
print(f"Meeting: {notes.title}")
print(f"Date: {notes.date}")
print(f"Attendees: {', '.join(notes.attendees)}")
print(f"\nTasks:")
for task in notes.tasks:
    print(f"  - [{task.priority.value}] {task.title} (assigned to: {task.assignee or 'unassigned'})")
print(f"\nSummary: {notes.summary}")

## 5. Classification with Confidence

Get AI to rate its confidence:

In [ ]:
class Classification(BaseModel):
    category: str
    confidence: float = Field(ge=0, le=1, description="Confidence score 0-1")
    reasoning: str

def classify_with_confidence(text: str, categories: List[str]) -> Classification:
    """Classify text into one of the categories with confidence."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Classify this text into one of these categories: {categories}

Text: {text}

Return JSON with:
- category: one of the categories above
- confidence: 0.0 to 1.0
- reasoning: brief explanation"""
        }],
        response_format={"type": "json_object"}
    )

    data = json.loads(response.choices[0].message.content)
    return Classification(**data)

# Test
texts = [
    "My order hasn't arrived yet, it's been 2 weeks!",
    "How do I reset my password?",
    "Your product is terrible and I want a refund",
    "Thanks for the quick delivery!"
]

categories = ["complaint", "question", "feedback", "other"]

for text in texts:
    result = classify_with_confidence(text, categories)
    print(f"'{text[:40]}...'")
    print(f"  → {result.category} ({result.confidence:.0%})")
    print(f"  Reason: {result.reasoning}")
    print()

## 6. Building a Data Extraction Pipeline

In [ ]:
class ContactInfo(BaseModel):
    name: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None
    company: Optional[str] = None

class DataExtractor:
    """Extract structured data from unstructured text."""

    def __init__(self):
        self.client = OpenAI()

    def extract(self, text: str, model_class: type) -> BaseModel:
        """Generic extraction for any Pydantic model."""

        # Get field descriptions
        fields = []
        for name, field in model_class.model_fields.items():
            fields.append(f"- {name}: {field.annotation.__name__}")

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": f"""Extract information from this text:

{text}

Return JSON with these fields (use null if not found):
{chr(10).join(fields)}"""
            }],
            response_format={"type": "json_object"}
        )

        data = json.loads(response.choices[0].message.content)
        return model_class(**data)

# Test
extractor = DataExtractor()

text = """Hi, this is Mark from TechCorp.
You can reach me at mark.wilson@techcorp.com or call 555-123-4567."""

contact = extractor.extract(text, ContactInfo)
print(f"Name: {contact.name}")
print(f"Email: {contact.email}")
print(f"Phone: {contact.phone}")
print(f"Company: {contact.company}")

## 7. Exercises

In [ ]:
# Exercise 1: Create a recipe extractor
# Extract ingredients and steps from recipe text

class Recipe(BaseModel):
    name: str
    servings: int
    ingredients: List[str]
    steps: List[str]

recipe_text = """
Simple Pasta (serves 4)

You'll need: 400g spaghetti, 2 cloves garlic, olive oil, salt, pepper, parmesan.

First, boil water and cook pasta until al dente. Meanwhile, sauté minced garlic
in olive oil. Drain pasta, toss with garlic oil, season with salt and pepper.
Serve with grated parmesan.
"""

# Your code here - use the DataExtractor pattern
# recipe = ...

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Define a Pydantic model for recipes:
- name: str
- ingredients: List[str]
- steps: List[str]
- prep_time_minutes: int
- difficulty: Literal["easy", "medium", "hard"]

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
from pydantic import BaseModel
from typing import List, Literal

class Recipe(BaseModel):
    name: str
    ingredients: List[str]
    steps: List[str]
    prep_time_minutes: int
    difficulty: Literal["easy", "medium", "hard"]

def extract_recipe(text: str) -> Recipe:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract recipe information."},
            {"role": "user", "content": text}
        ],
        response_format=Recipe
    )
    return response.choices[0].message.parsed

recipe = extract_recipe("To make pasta: boil water, add pasta, cook 8 mins...")
print(recipe.model_dump_json(indent=2))
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Define a Pydantic model for recipes:
- name: str
- ingredients: List[str]
- steps: List[str]
- prep_time_minutes: int
- difficulty: Literal["easy", "medium", "hard"]

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
from pydantic import BaseModel
from typing import List, Literal

class Recipe(BaseModel):
    name: str
    ingredients: List[str]
    steps: List[str]
    prep_time_minutes: int
    difficulty: Literal["easy", "medium", "hard"]

def extract_recipe(text: str) -> Recipe:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract recipe information."},
            {"role": "user", "content": text}
        ],
        response_format=Recipe
    )
    return response.choices[0].message.parsed

recipe = extract_recipe("To make pasta: boil water, add pasta, cook 8 mins...")
print(recipe.model_dump_json(indent=2))
```

</details>

In [ ]:
# Exercise 2: Multi-label classification
# Classify text into MULTIPLE categories

class MultiLabel(BaseModel):
    categories: List[str]
    confidence: float

def multi_classify(text: str):
    """Classify into multiple categories."""
    categories = ["urgent", "technical", "billing", "feedback", "general"]

    # Your implementation here
    pass

# Test with: "The app is crashing and I can't access my billing info! Please help ASAP!"

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

For multi-label, return a list of categories:
- Use List[str] in your model
- Or use a dict with boolean flags for each category

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class ContentLabels(BaseModel):
    categories: List[str]
    is_urgent: bool
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float

def classify_content(text: str) -> ContentLabels:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Classify the content."},
            {"role": "user", "content": text}
        ],
        response_format=ContentLabels
    )
    return response.choices[0].message.parsed

result = classify_content("URGENT: Server down! Need help immediately!")
print(result)  # categories=['technical', 'support'], is_urgent=True, ...
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

For multi-label, return a list of categories:
- Use List[str] in your model
- Or use a dict with boolean flags for each category

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class ContentLabels(BaseModel):
    categories: List[str]
    is_urgent: bool
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float

def classify_content(text: str) -> ContentLabels:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Classify the content."},
            {"role": "user", "content": text}
        ],
        response_format=ContentLabels
    )
    return response.choices[0].message.parsed

result = classify_content("URGENT: Server down! Need help immediately!")
print(result)  # categories=['technical', 'support'], is_urgent=True, ...
```

</details>

## Summary

You learned:
- ✅ How to make OpenAI API calls and understand tokens
- ✅ Message roles (system, user, assistant)
- ✅ Temperature, max_tokens parameters
- ✅ Building a chatbot with conversation memory
- ✅ Sliding window and summarization memory strategies
- ✅ Zero-shot, few-shot, and chain-of-thought prompting
- ✅ Getting reliable JSON with response_format
- ✅ Validating AI outputs with Pydantic

**Next:** [Week 2 - Connect to Your Data (RAG)](week_02_rag_intro.ipynb)